In [4]:
import numpy as np
from pathlib import Path

# Ajusta o path
aligned_dir = r"D:\User data\InesMarques\2photon\finalsemnan\aligned_CORRECT"

# Verificar ficheiros existentes
print("=== FICHEIROS DISPONÍVEIS ===")
for f in Path(aligned_dir).glob("*.npz"):
    print(f"  {f.name}")

# Carregar e mostrar estrutura
print("\n=== ESTRUTURA DOS DADOS ===")

# WITH_NAN
for fname in ["aligned_F_WITH_NAN.npz", "aligned_dffP20_WITH_NAN.npz"]:
    fpath = Path(aligned_dir) / fname
    if fpath.exists():
        data = np.load(fpath, allow_pickle=True)
        print(f"\n{fname}:")
        for key in data.files:
            arr = data[key]
            if isinstance(arr, np.ndarray):
                print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")
            else:
                print(f"  {key}: {type(arr)}")

# NO_BAD
for fname in ["aligned_F_NO_BAD.npz", "aligned_dffP20_NO_BAD.npz"]:
    fpath = Path(aligned_dir) / fname
    if fpath.exists():
        data = np.load(fpath, allow_pickle=True)
        print(f"\n{fname}:")
        for key in data.files:
            arr = data[key]
            if isinstance(arr, np.ndarray):
                print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")
            else:
                print(f"  {key}: {type(arr)}")

# Suite2p - verificar estrutura
suite2p_root = r"D:\User data\InesMarques\2photon\finalsemnan\suite2p_outputs"
print("\n=== SUITE2P STRUCTURE ===")

import os
import re

pattern = re.compile(r'aligned_p(\d+)_nan', re.IGNORECASE)
planes_found = []

for subdir in sorted(os.listdir(suite2p_root))[:5]:  # Primeiros 5
    full_path = os.path.join(suite2p_root, subdir)
    if os.path.isdir(full_path):
        match = pattern.match(subdir)
        if match:
            plane_num = int(match.group(1))
            suite2p_path = os.path.join(full_path, "suite2p", "plane0")
            
            if os.path.exists(suite2p_path):
                files = os.listdir(suite2p_path)
                print(f"  Plane {plane_num}: {subdir}/suite2p/plane0/")
                print(f"    Files: {files[:5]}...")  # Primeiros 5 ficheiros
                
                # Verificar shapes
                f_path = os.path.join(suite2p_path, "F.npy")
                stat_path = os.path.join(suite2p_path, "stat.npy")
                
                if os.path.exists(f_path):
                    F = np.load(f_path)
                    print(f"    F.npy: shape={F.shape}")
                
                if os.path.exists(stat_path):
                    stat = np.load(stat_path, allow_pickle=True)
                    print(f"    stat.npy: {len(stat)} ROIs")
                    if len(stat) > 0:
                        s = stat[0]
                        print(f"    stat[0] keys: {list(s.keys())[:10]}")
                
                planes_found.append(plane_num)

print(f"\n  Total planes found: {len(planes_found)} (showing first 5)")

# Verificar alinhamento: número de ROIs
print("\n=== VALIDAÇÃO ===")
data_f = np.load(Path(aligned_dir) / "aligned_F_WITH_NAN.npz")
n_rois_aligned = data_f['traces_F'].shape[0]
n_frames_aligned = data_f['traces_F'].shape[1]

print(f"  ROIs em aligned_F: {n_rois_aligned}")
print(f"  Frames em aligned_F: {n_frames_aligned}")
print(f"  Esperado: 180 planos × 250 fps = {180*250} frames")

# Contar ROIs no Suite2p
total_rois_suite2p = 0
for subdir in os.listdir(suite2p_root):
    full_path = os.path.join(suite2p_root, subdir)
    if os.path.isdir(full_path) and pattern.match(subdir):
        stat_path = os.path.join(full_path, "suite2p", "plane0", "stat.npy")
        if os.path.exists(stat_path):
            stat = np.load(stat_path, allow_pickle=True)
            total_rois_suite2p += len(stat)

print(f"  Total ROIs em Suite2p (todas): {total_rois_suite2p}")

=== FICHEIROS DISPONÍVEIS ===
  aligned_data_NO_BAD.npz
  aligned_data_WITH_NAN.npz
  aligned_dffP20_NO_BAD.npz
  aligned_dffP20_WITH_NAN.npz
  aligned_F_NO_BAD.npz
  aligned_F_WITH_NAN.npz

=== ESTRUTURA DOS DADOS ===

aligned_F_WITH_NAN.npz:
  traces_F: shape=(112252, 45000), dtype=float32
  regressors: shape=(45000, 68), dtype=float32
  frame_quality: shape=(45000,), dtype=bool
  regressor_names: shape=(68,), dtype=<U71
  description: shape=(), dtype=<U49

aligned_dffP20_WITH_NAN.npz:
  traces_dff: shape=(112252, 45000), dtype=float32
  regressors: shape=(45000, 68), dtype=float32
  frame_quality: shape=(45000,), dtype=bool
  regressor_names: shape=(68,), dtype=<U71
  description: shape=(), dtype=<U61

aligned_F_NO_BAD.npz:
  traces_F: shape=(112252, 42127), dtype=float32
  regressors: shape=(42127, 68), dtype=float32
  regressor_names: shape=(68,), dtype=<U71
  description: shape=(), dtype=<U28

aligned_dffP20_NO_BAD.npz:
  traces_dff: shape=(112252, 42127), dtype=float32
  regress

In [1]:
"""
=============================================================================
PIPELINE COMPLETO - VERIFICADO PARA ESTRUTURA DE NaN
=============================================================================

ESTRUTURA DOS DADOS:
- WITH_NAN: 45000 frames, NaN em:
  1. Frames de outros planos (estrutural)
  2. Bad frames dentro do plano (qualidade)
  
- NO_BAD: 42127 frames (variável por plano), sem NaN

Inês Marques - Janeiro 2025
=============================================================================
"""

import os
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


# =============================================================================
# CONFIG
# =============================================================================

class Config:
    ALIGNED_DATA_DIR = r"D:\User data\InesMarques\2photon\finalsemnan\aligned_CORRECT"
    SUITE2P_ROOT = r"D:\User data\InesMarques\2photon\finalsemnan\suite2p_outputs"
    NAN_JSON = r"D:\User data\InesMarques\2photon\finalsemnan\nan_frames_analysis.json"
    OUT_DIR = r"D:\User data\InesMarques\2photon\finalsemnan\ANALYSIS_READY"
    
    N_PLANES = 180
    FPS_PER_PLANE = 250
    
    AREA_MIN = 25
    AREA_MAX = 300
    ASPECT_MAX = 1.5
    
    OVERLAP_THR = 0.5
    CENTROID_MAX_DIST = 3.0
    MIN_SLICES = 2
    MAX_SLICES = 8
    
    CHUNK_SIZE = 5000

cfg = Config()


# =============================================================================
# SETUP
# =============================================================================

def setup_directories(base_dir):
    dirs = {
        'rois_2d': Path(base_dir) / 'rois_2d',
        'rois_3d': Path(base_dir) / 'rois_3d',
        'traces_2d': Path(base_dir) / 'traces_2d',
        'traces_3d': Path(base_dir) / 'traces_3d',
        'regressors': Path(base_dir) / 'regressors',
        'docs': Path(base_dir) / 'documentation',
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs

DIRS = setup_directories(cfg.OUT_DIR)


# =============================================================================
# NORMALIZAÇÕES
# =============================================================================

def center_traces(traces, chunk_size=5000):
    """Centra traces subtraindo média dos valores válidos."""
    n_rois = traces.shape[0]
    out = traces.copy()
    
    for start in range(0, n_rois, chunk_size):
        end = min(start + chunk_size, n_rois)
        for i in range(start, end):
            valid = ~np.isnan(out[i, :])
            if valid.sum() > 0:
                out[i, valid] -= np.nanmean(out[i, valid])
        
        if (end // chunk_size) % 5 == 0:
            print(f"      Centered {end:,}/{n_rois:,}")
    
    return out.astype(np.float32)


def zscore_traces(traces, chunk_size=5000):
    """Z-score por ROI sobre valores válidos."""
    n_rois = traces.shape[0]
    out = traces.copy()
    
    for start in range(0, n_rois, chunk_size):
        end = min(start + chunk_size, n_rois)
        for i in range(start, end):
            valid = ~np.isnan(out[i, :])
            if valid.sum() > 1:
                m = np.mean(out[i, valid])
                s = np.std(out[i, valid])
                if s > 1e-8:
                    out[i, valid] = (out[i, valid] - m) / s
                else:
                    out[i, valid] -= m
        
        if (end // chunk_size) % 5 == 0:
            print(f"      Z-scored {end:,}/{n_rois:,}")
    
    return out.astype(np.float32)


def zscore_regressors(regs):
    """Z-score por regressor ignorando NaN."""
    mean = np.nanmean(regs, axis=0, keepdims=True)
    std = np.nanstd(regs, axis=0, keepdims=True)
    std = np.maximum(std, 1e-8)
    return ((regs - mean) / std).astype(np.float32)


# =============================================================================
# LOAD & VALIDATE DATA
# =============================================================================

def load_aligned_data(aligned_dir):
    """Carrega e valida dados alinhados."""
    print("\n" + "="*70)
    print("LOADING ALIGNED DATA")
    print("="*70)
    
    p = Path(aligned_dir)
    
    f_nan = np.load(p / "aligned_F_WITH_NAN.npz", allow_pickle=True)
    dff_nan = np.load(p / "aligned_dffP20_WITH_NAN.npz", allow_pickle=True)
    f_nb = np.load(p / "aligned_F_NO_BAD.npz", allow_pickle=True)
    dff_nb = np.load(p / "aligned_dffP20_NO_BAD.npz", allow_pickle=True)
    
    data = {
        'F_with_nan': f_nan['traces_F'],
        'dff_with_nan': dff_nan['traces_dff'],
        'regs_with_nan': f_nan['regressors'],
        'frame_quality': f_nan['frame_quality'],
        'F_no_bad': f_nb['traces_F'],
        'dff_no_bad': dff_nb['traces_dff'],
        'regs_no_bad': f_nb['regressors'],
        'reg_names': list(f_nan['regressor_names']),
    }
    
    # === VALIDAÇÃO ===
    print("\n  [VALIDATION]")
    
    n_rois = data['F_with_nan'].shape[0]
    n_frames_nan = data['F_with_nan'].shape[1]
    n_frames_nb = data['F_no_bad'].shape[1]
    
    print(f"  ROIs: {n_rois:,}")
    print(f"  WITH_NAN frames: {n_frames_nan:,}")
    print(f"  NO_BAD frames: {n_frames_nb:,}")
    
    # Verificar NaN nos regressores
    regs_nan_count = np.isnan(data['regs_with_nan']).sum()
    regs_nb_nan_count = np.isnan(data['regs_no_bad']).sum()
    
    print(f"\n  Regressors WITH_NAN: {regs_nan_count:,} NaN values")
    print(f"  Regressors NO_BAD: {regs_nb_nan_count:,} NaN values")
    
    # Verificar consistência: NaN em regressores = bad frames
    bad_frames = ~data['frame_quality']
    n_bad = bad_frames.sum()
    
    # NaN em qualquer regressor
    regs_has_nan = np.any(np.isnan(data['regs_with_nan']), axis=1)
    
    print(f"\n  Bad frames (frame_quality=False): {n_bad:,}")
    print(f"  Frames with NaN in regressors: {regs_has_nan.sum():,}")
    
    # Verificar se coincidem
    if np.array_equal(bad_frames, regs_has_nan):
        print("  ✓ Regressors have NaN exactly in bad frames")
    else:
        print("  ⚠ Mismatch between bad frames and regressor NaN!")
    
    # Amostrar algumas ROIs para verificar traces
    print("\n  [SAMPLE ROI VALIDATION]")
    sample_rois = [0, n_rois//4, n_rois//2, 3*n_rois//4, n_rois-1]
    
    for roi_idx in sample_rois[:3]:
        trace = data['dff_with_nan'][roi_idx, :]
        n_valid = (~np.isnan(trace)).sum()
        n_nan = np.isnan(trace).sum()
        
        # Encontrar plano
        valid_indices = np.where(~np.isnan(trace))[0]
        if len(valid_indices) > 0:
            first_valid = valid_indices[0]
            plane = first_valid // cfg.FPS_PER_PLANE
            print(f"    ROI {roi_idx}: plane {plane}, {n_valid} valid, {n_nan} NaN")
    
    return data


def load_nan_info(json_path):
    """Carrega info de frames por plano."""
    print("\n[LOADING NaN INFO]")
    
    with open(json_path, 'r') as f:
        raw = json.load(f)
    
    frames_per_plane = {}
    
    for tif_name, details in raw.items():
        if details.get('is_single_frame', False) or 'error' in details:
            continue
        
        match = re.search(r'p(\d+)', tif_name)
        if not match:
            continue
        
        plane_idx = int(match.group(1)) - 1
        n_removed = len(details.get("removed_frames", []))
        n_interp = len(details.get("interpolated_frames", []))
        
        frames_per_plane[plane_idx] = {
            'n_original': cfg.FPS_PER_PLANE,
            'n_removed': n_removed,
            'n_interpolated': n_interp,
            'n_good': cfg.FPS_PER_PLANE - n_removed - n_interp,
            'bad_frames': details.get("removed_frames", []) + details.get("interpolated_frames", []),
        }
    
    # Calcular offsets para NO_BAD
    plane_offsets_nb = {}
    cumsum = 0
    for plane_idx in sorted(frames_per_plane.keys()):
        plane_offsets_nb[plane_idx] = cumsum
        cumsum += frames_per_plane[plane_idx]['n_good']
    
    print(f"  ✓ {len(frames_per_plane)} planes")
    print(f"  ✓ Total NO_BAD frames: {cumsum}")
    
    # Stats
    n_goods = [f['n_good'] for f in frames_per_plane.values()]
    print(f"  ✓ Frames/plane: min={min(n_goods)}, max={max(n_goods)}, mean={np.mean(n_goods):.1f}")
    
    return frames_per_plane, plane_offsets_nb


# =============================================================================
# ROI MAPPING
# =============================================================================

def build_roi_2d_mapping(suite2p_root):
    """Constrói mapeamento de ROIs 2D."""
    print("\n" + "="*70)
    print("BUILDING 2D ROI MAPPING")
    print("="*70)
    
    pattern = re.compile(r'aligned_p(\d+)_nan', re.IGNORECASE)
    plane_data = {}
    
    for subdir in sorted(os.listdir(suite2p_root)):
        full_path = os.path.join(suite2p_root, subdir)
        if not os.path.isdir(full_path):
            continue
        
        match = pattern.match(subdir)
        if match:
            plane_idx = int(match.group(1)) - 1
            stat_path = os.path.join(full_path, "suite2p", "plane0", "stat.npy")
            if os.path.exists(stat_path):
                plane_data[plane_idx] = np.load(stat_path, allow_pickle=True)
    
    print(f"  Found {len(plane_data)} planes")
    
    roi_2d_info = []
    global_idx = 0
    
    for plane_idx in sorted(plane_data.keys()):
        for local_idx, s in enumerate(plane_data[plane_idx]):
            ypix = np.asarray(s['ypix'], dtype=np.int16)
            xpix = np.asarray(s['xpix'], dtype=np.int16)
            area = len(ypix)
            
            if area > 0:
                cy = float(np.mean(ypix))
                cx = float(np.mean(xpix))
                h = ypix.max() - ypix.min() + 1
                w = xpix.max() - xpix.min() + 1
                aspect = max(h/w, w/h)
            else:
                cy, cx, aspect = np.nan, np.nan, np.inf
            
            roi_2d_info.append({
                'global_idx': global_idx,
                'plane_idx': plane_idx,
                'local_idx': local_idx,
                'area': area,
                'centroid_y': cy,
                'centroid_x': cx,
                'aspect_ratio': aspect,
                'ypix': ypix,
                'xpix': xpix,
            })
            global_idx += 1
    
    print(f"  ✓ Total ROIs: {len(roi_2d_info):,}")
    return roi_2d_info


def filter_rois_2d(roi_2d_info, cfg):
    """Filtra ROIs."""
    print("\n[FILTERING 2D ROIs]")
    
    filtered = [r['global_idx'] for r in roi_2d_info 
                if cfg.AREA_MIN <= r['area'] <= cfg.AREA_MAX 
                and r['aspect_ratio'] <= cfg.ASPECT_MAX]
    
    print(f"  Total: {len(roi_2d_info):,} → Kept: {len(filtered):,} ({100*len(filtered)/len(roi_2d_info):.1f}%)")
    return filtered


# =============================================================================
# STITCHING 3D
# =============================================================================

def find_stitching_edges(roi_2d_info, cfg):
    """Encontra pares de ROIs em planos adjacentes."""
    print("\n[FINDING STITCHING EDGES]")
    
    def pixset(r):
        return set(zip(r['ypix'].tolist(), r['xpix'].tolist()))
    
    def overlap(p1, p2):
        inter = len(p1 & p2)
        return inter / min(len(p1), len(p2)) if inter > 0 else 0.0
    
    rois_by_plane = defaultdict(list)
    for r in roi_2d_info:
        rois_by_plane[r['plane_idx']].append(r)
    
    edges = []
    planes = sorted(rois_by_plane.keys())
    
    for i in range(len(planes) - 1):
        z, z_next = planes[i], planes[i + 1]
        if z_next != z + 1:
            continue
        
        rois_A, rois_B = rois_by_plane[z], rois_by_plane[z_next]
        if not rois_A or not rois_B:
            continue
        
        A_pix = [pixset(r) for r in rois_A]
        B_pix = [pixset(r) for r in rois_B]
        A_c = np.array([(r['centroid_y'], r['centroid_x']) for r in rois_A])
        B_c = np.array([(r['centroid_y'], r['centroid_x']) for r in rois_B])
        
        best_AB, best_BA = {}, {}
        
        for ia in range(len(rois_A)):
            if not np.isfinite(A_c[ia, 0]):
                continue
            for ib in np.where(np.sqrt(((B_c - A_c[ia])**2).sum(1)) <= cfg.CENTROID_MAX_DIST)[0]:
                ov = overlap(A_pix[ia], B_pix[ib])
                if ov >= cfg.OVERLAP_THR:
                    if ia not in best_AB or ov > best_AB[ia][0]:
                        best_AB[ia] = (ov, ib)
                    if ib not in best_BA or ov > best_BA[ib][0]:
                        best_BA[ib] = (ov, ia)
        
        for ia, (_, ib) in best_AB.items():
            if ib in best_BA and best_BA[ib][1] == ia:
                edges.append((rois_A[ia]['global_idx'], rois_B[ib]['global_idx']))
    
    print(f"  ✓ Found {len(edges):,} edges")
    return edges


def build_3d_groups(roi_2d_info, edges, cfg):
    """Constrói grupos 3D."""
    print("\n[BUILDING 3D GROUPS]")
    
    parent = {r['global_idx']: r['global_idx'] for r in roi_2d_info}
    
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra
    
    comps = defaultdict(list)
    for r in roi_2d_info:
        comps[find(r['global_idx'])].append(r['global_idx'])
    
    idx_to_plane = {r['global_idx']: r['plane_idx'] for r in roi_2d_info}
    
    filtered = []
    for g in comps.values():
        planes = {idx_to_plane[idx] for idx in g}
        n = len(planes)
        
        if n < cfg.MIN_SLICES:
            continue
        
        if n <= cfg.MAX_SLICES:
            filtered.append(sorted(g))
        else:
            by_plane = defaultdict(list)
            for idx in g:
                by_plane[idx_to_plane[idx]].append(idx)
            sorted_planes = sorted(by_plane.keys())
            
            for k in range(0, len(sorted_planes), cfg.MAX_SLICES):
                chunk = sorted_planes[k:k + cfg.MAX_SLICES]
                if len(chunk) >= cfg.MIN_SLICES:
                    filtered.append(sorted([idx for p in chunk for idx in by_plane[p]]))
    
    print(f"  ✓ 3D groups: {len(filtered):,}")
    if filtered:
        slices = [len({idx_to_plane[i] for i in g}) for g in filtered]
        print(f"  Slices: {dict(Counter(slices))}")
    
    return filtered, idx_to_plane


# =============================================================================
# 3D TRACE RECONSTRUCTION
# =============================================================================

def reconstruct_3d_WITH_NAN(groups, idx_to_plane, traces_2d, cfg, method='feierstein'):
    """
    Reconstrói traces 3D para WITH_NAN.
    
    NOTA: traces_2d já tem NaN em:
    - Frames de outros planos
    - Bad frames dentro do plano
    
    np.nanmean ignora todos os NaN automaticamente.
    """
    print(f"\n[3D RECONSTRUCTION - WITH_NAN - {method}]")
    
    n_groups = len(groups)
    n_frames = traces_2d.shape[1]
    fps = cfg.FPS_PER_PLANE
    
    traces_3d = np.full((n_groups, n_frames), np.nan, dtype=np.float32)
    roi_3d_info = []
    
    for gid, g in enumerate(groups):
        by_plane = defaultdict(list)
        for idx in g:
            by_plane[idx_to_plane[idx]].append(idx)
        
        planes = sorted(by_plane.keys())
        
        if method == 'feierstein':
            # Feierstein: dF/F já normalizado → agregar → centrar entre planos
            plane_means = []
            
            for z in planes:
                start_f = z * fps
                end_f = start_f + fps
                if end_f > n_frames:
                    continue
                
                # nanmean ignora NaN (bad frames)
                plane_trace = np.nanmean(traces_2d[by_plane[z], start_f:end_f], axis=0)
                traces_3d[gid, start_f:end_f] = plane_trace
                
                # Média válida do plano
                valid_mean = np.nanmean(plane_trace)
                if np.isfinite(valid_mean):
                    plane_means.append(valid_mean)
            
            # Centrar subtraindo média entre planos
            if len(plane_means) > 0:
                global_mean = np.mean(plane_means)
                for z in planes:
                    start_f = z * fps
                    end_f = start_f + fps
                    if end_f <= n_frames:
                        valid = ~np.isnan(traces_3d[gid, start_f:end_f])
                        traces_3d[gid, start_f:end_f][valid] -= global_mean
        
        else:  # aggregate_first
            for z in planes:
                start_f = z * fps
                end_f = start_f + fps
                if end_f > n_frames:
                    continue
                traces_3d[gid, start_f:end_f] = np.nanmean(
                    traces_2d[by_plane[z], start_f:end_f], axis=0
                )
        
        roi_3d_info.append({
            'roi_3d_id': gid,
            'member_2d_indices': list(g),
            'n_2d_rois': len(g),
            'n_slices': len(planes),
            'z_planes': planes,
            'z_start': min(planes),
            'z_end': max(planes),
        })
        
        if (gid + 1) % 10000 == 0:
            print(f"    {gid + 1:,}/{n_groups:,}")
    
    # Estatísticas
    n_valid = np.sum(~np.isnan(traces_3d))
    n_total = traces_3d.size
    print(f"  ✓ Shape: {traces_3d.shape}")
    print(f"  ✓ Valid values: {n_valid:,}/{n_total:,} ({100*n_valid/n_total:.2f}%)")
    
    return traces_3d, roi_3d_info


def reconstruct_3d_NO_BAD(groups, idx_to_plane, traces_2d, frames_per_plane, plane_offsets, method='feierstein'):
    """
    Reconstrói traces 3D para NO_BAD.
    
    NOTA: NO_BAD não tem bad frames, mas cada plano tem número diferente de frames.
    Cada ROI ainda só tem valores no seu segmento.
    """
    print(f"\n[3D RECONSTRUCTION - NO_BAD - {method}]")
    
    n_groups = len(groups)
    n_frames = traces_2d.shape[1]
    
    traces_3d = np.full((n_groups, n_frames), np.nan, dtype=np.float32)
    
    for gid, g in enumerate(groups):
        by_plane = defaultdict(list)
        for idx in g:
            by_plane[idx_to_plane[idx]].append(idx)
        
        planes = sorted(by_plane.keys())
        
        if method == 'feierstein':
            plane_means = []
            
            for z in planes:
                if z not in plane_offsets or z not in frames_per_plane:
                    continue
                
                start_f = plane_offsets[z]
                n_good = frames_per_plane[z]['n_good']
                end_f = start_f + n_good
                
                if end_f > n_frames:
                    continue
                
                plane_trace = np.nanmean(traces_2d[by_plane[z], start_f:end_f], axis=0)
                traces_3d[gid, start_f:end_f] = plane_trace
                
                valid_mean = np.nanmean(plane_trace)
                if np.isfinite(valid_mean):
                    plane_means.append(valid_mean)
            
            if len(plane_means) > 0:
                global_mean = np.mean(plane_means)
                for z in planes:
                    if z not in plane_offsets:
                        continue
                    start_f = plane_offsets[z]
                    n_good = frames_per_plane[z]['n_good']
                    end_f = start_f + n_good
                    if end_f <= n_frames:
                        valid = ~np.isnan(traces_3d[gid, start_f:end_f])
                        traces_3d[gid, start_f:end_f][valid] -= global_mean
        
        else:
            for z in planes:
                if z not in plane_offsets or z not in frames_per_plane:
                    continue
                start_f = plane_offsets[z]
                n_good = frames_per_plane[z]['n_good']
                end_f = start_f + n_good
                if end_f > n_frames:
                    continue
                traces_3d[gid, start_f:end_f] = np.nanmean(
                    traces_2d[by_plane[z], start_f:end_f], axis=0
                )
        
        if (gid + 1) % 10000 == 0:
            print(f"    {gid + 1:,}/{n_groups:,}")
    
    n_valid = np.sum(~np.isnan(traces_3d))
    print(f"  ✓ Shape: {traces_3d.shape}")
    print(f"  ✓ Valid: {n_valid:,}/{traces_3d.size:,}")
    
    return traces_3d


# =============================================================================
# SAVE FUNCTIONS
# =============================================================================

def save_2d_data(data, roi_2d_info, filtered_indices, cfg):
    """Salva dados 2D."""
    print("\n" + "="*70)
    print("SAVING 2D DATA")
    print("="*70)
    
    frame_quality = data['frame_quality']
    
    # WITH_NAN
    print("\n  [WITH_NAN]")
    for name, traces_raw, desc in [
        ("dff", data['dff_with_nan'], "dF/F (P20)"),
        ("F", data['F_with_nan'], "Raw F"),
    ]:
        np.savez_compressed(
            DIRS['traces_2d'] / f"traces_2d_{name}_WITH_NAN.npz",
            traces=traces_raw,
            frame_quality=frame_quality,
            description=desc
        )
        print(f"    ✓ traces_2d_{name}_WITH_NAN.npz")
    
    print("    Computing dff_centered...")
    dff_centered = center_traces(data['dff_with_nan'], cfg.CHUNK_SIZE)
    np.savez_compressed(
        DIRS['traces_2d'] / "traces_2d_dff_centered_WITH_NAN.npz",
        traces=dff_centered,
        frame_quality=frame_quality,
        description="dF/F centered (Feierstein)"
    )
    print(f"    ✓ traces_2d_dff_centered_WITH_NAN.npz")
    del dff_centered
    
    print("    Computing zscore...")
    zscore = zscore_traces(data['dff_with_nan'], cfg.CHUNK_SIZE)
    np.savez_compressed(
        DIRS['traces_2d'] / "traces_2d_zscore_WITH_NAN.npz",
        traces=zscore,
        frame_quality=frame_quality,
        description="Z-score of dF/F"
    )
    print(f"    ✓ traces_2d_zscore_WITH_NAN.npz")
    del zscore
    
    # NO_BAD
    print("\n  [NO_BAD]")
    for name, traces_raw, desc in [
        ("dff", data['dff_no_bad'], "dF/F (P20)"),
        ("F", data['F_no_bad'], "Raw F"),
    ]:
        np.savez_compressed(
            DIRS['traces_2d'] / f"traces_2d_{name}_NO_BAD.npz",
            traces=traces_raw,
            description=desc
        )
        print(f"    ✓ traces_2d_{name}_NO_BAD.npz")
    
    print("    Computing dff_centered...")
    dff_centered = center_traces(data['dff_no_bad'], cfg.CHUNK_SIZE)
    np.savez_compressed(
        DIRS['traces_2d'] / "traces_2d_dff_centered_NO_BAD.npz",
        traces=dff_centered,
        description="dF/F centered"
    )
    print(f"    ✓ traces_2d_dff_centered_NO_BAD.npz")
    del dff_centered
    
    print("    Computing zscore...")
    zscore = zscore_traces(data['dff_no_bad'], cfg.CHUNK_SIZE)
    np.savez_compressed(
        DIRS['traces_2d'] / "traces_2d_zscore_NO_BAD.npz",
        traces=zscore,
        description="Z-score of dF/F"
    )
    print(f"    ✓ traces_2d_zscore_NO_BAD.npz")
    del zscore
    
    # Metadata
    print("\n  [Metadata]")
    df = pd.DataFrame([{
        'global_idx': r['global_idx'],
        'plane_idx': r['plane_idx'],
        'local_idx': r['local_idx'],
        'area': r['area'],
        'centroid_y': r['centroid_y'],
        'centroid_x': r['centroid_x'],
        'aspect_ratio': r['aspect_ratio'],
        'passed_filter': r['global_idx'] in filtered_indices,
    } for r in roi_2d_info])
    df.to_csv(DIRS['rois_2d'] / "roi_2d_metadata.csv", index=False)
    print(f"    ✓ roi_2d_metadata.csv")
    
    np.savez_compressed(
        DIRS['rois_2d'] / "roi_2d_coordinates.npz",
        ypix=np.array([r['ypix'] for r in roi_2d_info], dtype=object),
        xpix=np.array([r['xpix'] for r in roi_2d_info], dtype=object),
        plane_idx=np.array([r['plane_idx'] for r in roi_2d_info]),
        centroid_y=np.array([r['centroid_y'] for r in roi_2d_info]),
        centroid_x=np.array([r['centroid_x'] for r in roi_2d_info]),
    )
    print(f"    ✓ roi_2d_coordinates.npz")
    
    np.save(DIRS['rois_2d'] / "filtered_indices.npy", np.array(filtered_indices))
    print(f"    ✓ filtered_indices.npy")


def save_3d_data(groups, idx_to_plane, roi_2d_info, data, frames_per_plane, plane_offsets_nb, cfg):
    """Salva dados 3D com múltiplos métodos."""
    print("\n" + "="*70)
    print("SAVING 3D DATA")
    print("="*70)
    
    if not groups:
        print("  ⚠ No 3D ROIs")
        return []
    
    frame_quality = data['frame_quality']
    
    # === FEIERSTEIN METHOD ===
    print("\n  [FEIERSTEIN METHOD]")
    
    traces_fei_nan, roi_3d_info = reconstruct_3d_WITH_NAN(
        groups, idx_to_plane, data['dff_with_nan'], cfg, 'feierstein'
    )
    np.savez_compressed(
        DIRS['traces_3d'] / "traces_3d_feierstein_WITH_NAN.npz",
        traces=traces_fei_nan,
        frame_quality=frame_quality,
        description="Feierstein: dF/F 2D → aggregate → center across planes"
    )
    print(f"    ✓ traces_3d_feierstein_WITH_NAN.npz")
    
    # NO_BAD - só retorna traces, não roi_3d_info
    traces_fei_nb = reconstruct_3d_NO_BAD(
        groups, idx_to_plane, data['dff_no_bad'], frames_per_plane, plane_offsets_nb, 'feierstein'
    )
    np.savez_compressed(
        DIRS['traces_3d'] / "traces_3d_feierstein_NO_BAD.npz",
        traces=traces_fei_nb,
        description="Feierstein method, no bad frames"
    )
    print(f"    ✓ traces_3d_feierstein_NO_BAD.npz")
    
    del traces_fei_nan, traces_fei_nb
    
    # === AGGREGATE THEN NORMALIZE ===
    print("\n  [AGGREGATE THEN NORMALIZE]")
    
    # WITH_NAN - retorna (traces, roi_info) mas já temos roi_3d_info
    traces_agg_nan, _ = reconstruct_3d_WITH_NAN(
        groups, idx_to_plane, data['dff_with_nan'], cfg, 'aggregate_first'
    )
    
    # Centered
    traces_agg_c = center_traces(traces_agg_nan, cfg.CHUNK_SIZE)
    np.savez_compressed(
        DIRS['traces_3d'] / "traces_3d_agg_centered_WITH_NAN.npz",
        traces=traces_agg_c,
        frame_quality=frame_quality,
        description="Aggregate dF/F then center"
    )
    print(f"    ✓ traces_3d_agg_centered_WITH_NAN.npz")
    del traces_agg_c
    
    # Z-score
    traces_agg_z = zscore_traces(traces_agg_nan, cfg.CHUNK_SIZE)
    np.savez_compressed(
        DIRS['traces_3d'] / "traces_3d_agg_zscore_WITH_NAN.npz",
        traces=traces_agg_z,
        frame_quality=frame_quality,
        description="Aggregate dF/F then Z-score"
    )
    print(f"    ✓ traces_3d_agg_zscore_WITH_NAN.npz")
    del traces_agg_z, traces_agg_nan
    
    # NO_BAD versions - CORRIGIDO: só retorna traces!
    traces_agg_nb = reconstruct_3d_NO_BAD(
        groups, idx_to_plane, data['dff_no_bad'], frames_per_plane, plane_offsets_nb, 'aggregate_first'
    )
    
    traces_agg_c_nb = center_traces(traces_agg_nb, cfg.CHUNK_SIZE)
    np.savez_compressed(
        DIRS['traces_3d'] / "traces_3d_agg_centered_NO_BAD.npz",
        traces=traces_agg_c_nb,
        description="Aggregate then center, no bad"
    )
    print(f"    ✓ traces_3d_agg_centered_NO_BAD.npz")
    del traces_agg_c_nb
    
    traces_agg_z_nb = zscore_traces(traces_agg_nb, cfg.CHUNK_SIZE)
    np.savez_compressed(
        DIRS['traces_3d'] / "traces_3d_agg_zscore_NO_BAD.npz",
        traces=traces_agg_z_nb,
        description="Aggregate then Z-score, no bad"
    )
    print(f"    ✓ traces_3d_agg_zscore_NO_BAD.npz")
    del traces_agg_z_nb, traces_agg_nb
    
    # === METADATA & COORDINATES === (resto igual...)
    
    # === METADATA & COORDINATES ===
    print("\n  [Metadata & Coordinates]")
    
    idx_to_2d = {r['global_idx']: r for r in roi_2d_info}
    
    df = pd.DataFrame([{
        'roi_3d_id': r['roi_3d_id'],
        'n_2d_rois': r['n_2d_rois'],
        'n_slices': r['n_slices'],
        'z_start': r['z_start'],
        'z_end': r['z_end'],
    } for r in roi_3d_info])
    df.to_csv(DIRS['rois_3d'] / "roi_3d_metadata.csv", index=False)
    print(f"    ✓ roi_3d_metadata.csv")
    
    with open(DIRS['rois_3d'] / "roi_3d_to_2d_mapping.json", 'w') as f:
        json.dump({r['roi_3d_id']: r['member_2d_indices'] for r in roi_3d_info}, f)
    print(f"    ✓ roi_3d_to_2d_mapping.json")
    
    # 3D Coordinates
    coords_3d = []
    for r in roi_3d_info:
        all_y, all_x, all_z = [], [], []
        centroids = []
        
        for idx_2d in r['member_2d_indices']:
            r2d = idx_to_2d[idx_2d]
            z = r2d['plane_idx']
            all_y.extend(r2d['ypix'].tolist())
            all_x.extend(r2d['xpix'].tolist())
            all_z.extend([z] * len(r2d['ypix']))
            centroids.append((r2d['centroid_y'], r2d['centroid_x'], z))
        
        coords_3d.append({
            'ypix': np.array(all_y, dtype=np.int16),
            'xpix': np.array(all_x, dtype=np.int16),
            'zpix': np.array(all_z, dtype=np.int16),
            'centroid_y': np.mean([c[0] for c in centroids]),
            'centroid_x': np.mean([c[1] for c in centroids]),
            'centroid_z': np.mean([c[2] for c in centroids]),
        })
    
    np.savez_compressed(
        DIRS['rois_3d'] / "roi_3d_coordinates.npz",
        ypix=np.array([c['ypix'] for c in coords_3d], dtype=object),
        xpix=np.array([c['xpix'] for c in coords_3d], dtype=object),
        zpix=np.array([c['zpix'] for c in coords_3d], dtype=object),
        centroid_y=np.array([c['centroid_y'] for c in coords_3d]),
        centroid_x=np.array([c['centroid_x'] for c in coords_3d]),
        centroid_z=np.array([c['centroid_z'] for c in coords_3d]),
    )
    print(f"    ✓ roi_3d_coordinates.npz")
    
    return roi_3d_info


def save_regressors(data):
    """Salva regressores."""
    print("\n" + "="*70)
    print("SAVING REGRESSORS")
    print("="*70)
    
    for suffix, regs, fq in [
        ("WITH_NAN", data['regs_with_nan'], data['frame_quality']),
        ("NO_BAD", data['regs_no_bad'], None),
    ]:
        regs_z = zscore_regressors(regs)
        
        save_dict = {
            'regressors_raw': regs,
            'regressors_zscore': regs_z,
            'reg_names': np.array(data['reg_names']),
        }
        if fq is not None:
            save_dict['frame_quality'] = fq
        
        np.savez_compressed(DIRS['regressors'] / f"regressors_{suffix}.npz", **save_dict)
        print(f"  ✓ regressors_{suffix}.npz: {regs.shape}")
    
    with open(DIRS['regressors'] / "regressor_names.txt", 'w') as f:
        for i, name in enumerate(data['reg_names']):
            f.write(f"{i}\t{name}\n")
    print(f"  ✓ regressor_names.txt")


def save_documentation(data, roi_2d_info, roi_3d_info, cfg):
    """Documentação."""
    print("\n" + "="*70)
    print("SAVING DOCUMENTATION")
    print("="*70)
    
    doc = f"""
================================================================================
ANALYSIS READY - COMPLETE DOCUMENTATION
================================================================================

Created: {datetime.now().strftime('%Y-%m-%d %H:%M')}

================================================================================
DATA STRUCTURE
================================================================================

WITH_NAN (45000 frames = 180 planes × 250):
- Each ROI has ~230-250 valid values (only in its plane)
- NaN in: (1) frames of other planes, (2) bad frames within plane
- Regressors ALSO have NaN in bad frames

NO_BAD (42127 frames):
- Bad frames removed globally
- Each plane has variable number of frames (~230-250)
- Each ROI still only has values in its plane segment
- NO NaN in regressors

================================================================================
3D AGGREGATION METHODS
================================================================================

FEIERSTEIN (traces_3d_feierstein_*.npz) ← RECOMMENDED
- Based on Feierstein et al. (2023) Current Biology
- dF/F normalized per 2D ROI (P20 baseline)
- Aggregate across planes (nanmean ignores bad frames)
- Center by subtracting mean across planes

AGG_CENTERED (traces_3d_agg_centered_*.npz)
- Aggregate dF/F first
- Then center globally

AGG_ZSCORE (traces_3d_agg_zscore_*.npz)
- Aggregate dF/F first
- Then Z-score globally

================================================================================
FILES
================================================================================

traces_2d/
├── traces_2d_dff_WITH_NAN.npz           # dF/F raw
├── traces_2d_dff_centered_WITH_NAN.npz  # dF/F centered ← RECOMMENDED 2D
├── traces_2d_zscore_WITH_NAN.npz        # Z-score
├── traces_2d_F_WITH_NAN.npz             # Raw F
└── [same _NO_BAD versions]

traces_3d/
├── traces_3d_feierstein_WITH_NAN.npz    # ← RECOMMENDED 3D
├── traces_3d_feierstein_NO_BAD.npz
├── traces_3d_agg_centered_*.npz
└── traces_3d_agg_zscore_*.npz

rois_2d/
├── roi_2d_metadata.csv
├── roi_2d_coordinates.npz
└── filtered_indices.npy

rois_3d/
├── roi_3d_metadata.csv
├── roi_3d_coordinates.npz   # WITH zpix for 3D visualization!
└── roi_3d_to_2d_mapping.json

regressors/
├── regressors_WITH_NAN.npz  # Has NaN in bad frames
├── regressors_NO_BAD.npz    # No NaN
└── regressor_names.txt

================================================================================
CORRELATION EXAMPLE
================================================================================
```python
# Load
traces = np.load('traces_2d/traces_2d_dff_centered_WITH_NAN.npz')['traces']
regs_data = np.load('regressors/regressors_WITH_NAN.npz')
regs = regs_data['regressors_zscore']

# For each ROI
for i in range(n_rois):
    # Valid = not NaN in BOTH trace and regressors
    valid = ~np.isnan(traces[i, :]) & ~np.isnan(regs[:, 0])
    
    if valid.sum() < 50:
        continue
    
    for j in range(n_regs):
        corr[i, j] = np.corrcoef(traces[i, valid], regs[valid, j])[0, 1]

# Filter as Feierstein (|r| > 0.4)
max_corr = np.nanmax(np.abs(corr), axis=1)
good_rois = np.where(max_corr > 0.4)[0]
```

================================================================================
"""
    
    with open(DIRS['docs'] / "README.txt", 'w') as f:
        f.write(doc)
    print(f"  ✓ README.txt")
    
    summary = {
        "n_rois_2d": len(roi_2d_info),
        "n_rois_3d": len(roi_3d_info),
        "n_regressors": len(data['reg_names']),
        "frames_with_nan": int(data['dff_with_nan'].shape[1]),
        "frames_no_bad": int(data['dff_no_bad'].shape[1]),
    }
    with open(DIRS['docs'] / "summary.json", 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"  ✓ summary.json")


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("="*80)
    print("COMPLETE PIPELINE")
    print("="*80)
    
    # Load
    data = load_aligned_data(cfg.ALIGNED_DATA_DIR)
    frames_per_plane, plane_offsets_nb = load_nan_info(cfg.NAN_JSON)
    
    # ROI mapping
    roi_2d_info = build_roi_2d_mapping(cfg.SUITE2P_ROOT)
    
    if data['dff_with_nan'].shape[0] != len(roi_2d_info):
        print(f"⚠ ROI count mismatch!")
        return
    
    # Filter
    filtered_indices = filter_rois_2d(roi_2d_info, cfg)
    
    # Stitching
    edges = find_stitching_edges(roi_2d_info, cfg)
    groups, idx_to_plane = build_3d_groups(roi_2d_info, edges, cfg)
    
    # Save
    save_2d_data(data, roi_2d_info, filtered_indices, cfg)
    roi_3d_info = save_3d_data(groups, idx_to_plane, roi_2d_info, data, 
                                frames_per_plane, plane_offsets_nb, cfg)
    save_regressors(data)
    save_documentation(data, roi_2d_info, roi_3d_info, cfg)
    
    print("\n" + "="*80)
    print("✅ COMPLETE")
    print("="*80)
    print(f"  2D ROIs: {len(roi_2d_info):,}")
    print(f"  3D ROIs: {len(roi_3d_info):,}")
    print(f"  Output: {cfg.OUT_DIR}")
    print("="*80)


if __name__ == "__main__":
    main()

COMPLETE PIPELINE

LOADING ALIGNED DATA

  [VALIDATION]
  ROIs: 112,252
  WITH_NAN frames: 45,000
  NO_BAD frames: 42,127

  Regressors WITH_NAN: 195,364 NaN values
  Regressors NO_BAD: 0 NaN values

  Bad frames (frame_quality=False): 2,873
  Frames with NaN in regressors: 2,873
  ✓ Regressors have NaN exactly in bad frames

  [SAMPLE ROI VALIDATION]
    ROI 0: plane 0, 236 valid, 44764 NaN
    ROI 28063: plane 57, 243 valid, 44757 NaN
    ROI 56126: plane 91, 236 valid, 44764 NaN

[LOADING NaN INFO]
  ✓ 180 planes
  ✓ Total NO_BAD frames: 42127
  ✓ Frames/plane: min=221, max=249, mean=234.0

BUILDING 2D ROI MAPPING
  Found 180 planes
  ✓ Total ROIs: 112,252

[FILTERING 2D ROIs]
  Total: 112,252 → Kept: 107,947 (96.2%)

[FINDING STITCHING EDGES]
  ✓ Found 86,999 edges

[BUILDING 3D GROUPS]
  ✓ 3D groups: 19,991
  Slices: {3: 2795, 2: 4178, 4: 2183, 8: 6066, 5: 1853, 7: 1375, 6: 1541}

SAVING 2D DATA

  [WITH_NAN]
    ✓ traces_2d_dff_WITH_NAN.npz
    ✓ traces_2d_F_WITH_NAN.npz
    Comp

UnicodeEncodeError: 'charmap' codec can't encode character '\u2190' in position 1055: character maps to <undefined>

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from datetime import datetime

base = Path(r"D:\User data\InesMarques\2photon\finalsemnan\ANALYSIS_READY")
docs_dir = base / "documentation"
docs_dir.mkdir(parents=True, exist_ok=True)

roi2d = pd.read_csv(base / "rois_2d" / "roi_2d_metadata.csv")
roi3d = pd.read_csv(base / "rois_3d" / "roi_3d_metadata.csv")

regs = np.load(base / "regressors" / "regressors_WITH_NAN.npz", allow_pickle=True)
n_reg = int(len(regs["reg_names"]))
frames_with_nan = int(regs["regressors_raw"].shape[0])

regs2 = np.load(base / "regressors" / "regressors_NO_BAD.npz", allow_pickle=True)
frames_no_bad = int(regs2["regressors_raw"].shape[0])

doc = f"""
================================================================================
ANALYSIS READY  COMPLETE DOCUMENTATION
================================================================================

Created  {datetime.now().strftime('%Y-%m-%d %H:%M')}

2D ROIs  {len(roi2d)}
3D ROIs  {len(roi3d)}
Regressors  {n_reg}
Frames WITH_NAN  {frames_with_nan}
Frames NO_BAD  {frames_no_bad}
"""

with open(docs_dir / "README.txt", "w", encoding="utf-8", newline="\n") as f:
    f.write(doc)

summary = {
    "n_rois_2d": int(len(roi2d)),
    "n_rois_3d": int(len(roi3d)),
    "n_regressors": int(n_reg),
    "frames_with_nan": int(frames_with_nan),
    "frames_no_bad": int(frames_no_bad),
}
with open(docs_dir / "summary.json", "w", encoding="utf-8", newline="\n") as f:
    json.dump(summary, f, indent=2)

print("✓ README.txt and summary.json regenerated without rerunning the pipeline")


✓ README.txt and summary.json regenerated without rerunning the pipeline
